# 06 — Xception 299 + Fold-0 Diversity Screen

Mục tiêu: thử Xception pretrained ở resolution 299 và đo xem các model mắc lỗi giống hay khác nhau trên cùng fold 0.

- Holdout 770 ảnh vẫn niêm phong.
- Xception candidate: `xception41p.ra3_in1k` từ timm, eval size 299.
- So sánh với DenseNet121, ConvNeXt-Tiny, EfficientNet-B2.
- Tạo correlation/error-overlap và thử ensemble weights trên fold 0 chỉ để screening.

- Memory-safe config: physical batch 4 + gradient accumulation 2 (effective batch 8).

- Speed fix: preload pad+resize results into ~1.1 GiB RAM cache so each epoch does not repeatedly decode/resize full-resolution JPEGs.


In [ ]:
import os
os.environ['TORCH_HOME']=r'D:\AI_Cache\torch'
os.environ['HF_HOME']=r'D:\AI_Cache\huggingface'

from pathlib import Path
import json, time, random, gc
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score

try:
    import timm
except ImportError as e:
    raise RuntimeError('Missing timm. Run in Terminal: python -m pip install timm') from e

SEED=2026
ROOT=Path(r'D:\medicine')
MANIFEST=ROOT/'split_outputs'/'split_manifest.csv'
OUT=ROOT/'screen_outputs'/'xception_diversity'
OUT.mkdir(parents=True,exist_ok=True)
FOLD=0; IMAGE_SIZE=299; BATCH_SIZE=4; GRAD_ACCUM_STEPS=2; EPOCHS=7; PATIENCE=2
CACHE_RESIZED_IN_RAM=True; PROGRESS_EVERY=100
LR=1e-4; WD=1e-4; NUM_WORKERS=0
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP=DEVICE.type=='cuda'
torch.backends.cudnn.benchmark=USE_AMP
print('Device:',DEVICE)
if USE_AMP: print('GPU:',torch.cuda.get_device_name(0))
print('timm:',timm.__version__)
print('Batch size:',BATCH_SIZE,'| grad accumulation:',GRAD_ACCUM_STEPS,'| effective batch:',BATCH_SIZE*GRAD_ACCUM_STEPS)


In [ ]:
m=pd.read_csv(MANIFEST)
dev=m[m.split=='dev'].copy(); holdout=m[m.split=='holdout'].copy()
train_df=dev[dev.cv_fold!=FOLD].copy().reset_index(drop=True)
val_df=dev[dev.cv_fold==FOLD].copy().reset_index(drop=True)
assert len(train_df)==3489 and len(val_df)==873 and len(holdout)==770
print('Train:',len(train_df),'Val:',len(val_df),'SEALED holdout:',len(holdout))


In [ ]:
class PadToSquare:
    def __call__(self,img):
        w,h=img.size; s=max(w,h)
        l=(s-w)//2; r=s-w-l; t=(s-h)//2; b=s-h-t
        return ImageOps.expand(img,border=(l,t,r,b),fill=0)

MEAN=[0.485,0.456,0.406]; STD=[0.229,0.224,0.225]
augment_tf=transforms.Compose([
    transforms.RandomAffine(degrees=5,translate=(0.02,0.02),scale=(0.95,1.05),fill=0),
    transforms.ColorJitter(brightness=0.08,contrast=0.08),
    transforms.ToTensor(),transforms.Normalize(MEAN,STD)])
eval_tf=transforms.Compose([transforms.ToTensor(),transforms.Normalize(MEAN,STD)])

L2I={'Normal':0,'Pneumonia':1}

def preload_resized(frame, size=IMAGE_SIZE):
    arr=[]
    pad=PadToSquare()
    print(f'Preloading {len(frame)} images -> {size}x{size} uint8 RAM cache...')
    t0=time.time()
    for i,row in frame.reset_index(drop=True).iterrows():
        with Image.open(row.path) as im:
            im=im.convert('RGB'); im=pad(im); im=im.resize((size,size),Image.Resampling.BILINEAR)
            arr.append(np.asarray(im,dtype=np.uint8))
        if (i+1)%500==0 or (i+1)==len(frame): print(f'  cached {i+1}/{len(frame)}')
    out=np.stack(arr,axis=0)
    print(f'Cache shape={out.shape}, RAM={out.nbytes/1024**3:.2f} GiB, time={time.time()-t0:.0f}s')
    return out

train_cache=preload_resized(train_df) if CACHE_RESIZED_IN_RAM else None
val_cache=preload_resized(val_df) if CACHE_RESIZED_IN_RAM else None

class XrayDataset(Dataset):
    def __init__(self,df,tf,cache=None): self.df=df.reset_index(drop=True); self.tf=tf; self.cache=cache
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        row=self.df.iloc[i]
        if self.cache is not None:
            im=Image.fromarray(self.cache[i])
        else:
            with Image.open(row.path) as x:
                x=x.convert('RGB'); x=PadToSquare()(x); im=x.resize((IMAGE_SIZE,IMAGE_SIZE),Image.Resampling.BILINEAR)
        return self.tf(im),L2I[row.label],row.filename

train_loader=DataLoader(XrayDataset(train_df,augment_tf,train_cache),batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=False)
val_loader=DataLoader(XrayDataset(val_df,eval_tf,val_cache),batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=False)


In [ ]:
def metric(y,prob,t=0.5):
    y=np.asarray(y,int); prob=np.asarray(prob,float); pred=(prob>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    acc=(tp+tn)/len(y); se=tp/max(tp+fn,1); sp=tn/max(tn+fp,1)
    f1=f1_score(y,pred,pos_label=1,zero_division=0)
    P=.30*acc+.25*se+.20*sp+.25*f1
    return {'P':float(P),'Accuracy':float(acc),'Sensitivity':float(se),'Specificity':float(sp),'F1':float(f1),'AUROC':float(roc_auc_score(y,prob)),'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),'threshold':float(t)}

@torch.no_grad()
def predict(model,loader):
    model.eval(); ys=[]; ps=[]; names=[]
    for x,y,n in loader:
        x=x.to(DEVICE,non_blocking=True)
        with torch.amp.autocast(device_type='cuda',enabled=USE_AMP): logits=model(x)
        p=torch.softmax(logits.float(),1)[:,1]
        ys.extend(y.numpy()); ps.extend(p.cpu().numpy()); names.extend(list(n))
    return np.asarray(ys),np.asarray(ps),names


## Train Xception41p @ 299

In [ ]:
gc.collect()
if USE_AMP: torch.cuda.empty_cache()
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if USE_AMP: torch.cuda.manual_seed_all(SEED)
model=timm.create_model('xception41p.ra3_in1k',pretrained=True,num_classes=2).to(DEVICE)
c=train_df.label.value_counts(); n0=int(c.Normal); n1=int(c.Pneumonia); total=n0+n1
cw=torch.tensor([total/(2*n0),total/(2*n1)],dtype=torch.float32,device=DEVICE)
criterion=nn.CrossEntropyLoss(weight=cw)
optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WD)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS)
scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP)
best=-1; best_state=None; best_epoch=0; stale=0; hist=[]
for epoch in range(1,EPOCHS+1):
    t0=time.time(); model.train(); loss_sum=0; seen=0
    for batch_idx,(x,y,_) in enumerate(train_loader,1):
        x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
        if seen % (BATCH_SIZE*GRAD_ACCUM_STEPS) == 0:
            optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type='cuda',enabled=USE_AMP):
            logits=model(x); raw_loss=criterion(logits,y); loss=raw_loss/GRAD_ACCUM_STEPS
        scaler.scale(loss).backward()
        is_step = (((seen//BATCH_SIZE)+1) % GRAD_ACCUM_STEPS == 0) or (seen + len(x) >= len(train_loader.dataset))
        if is_step:
            scaler.step(optimizer); scaler.update()
        loss_sum+=float(raw_loss.item())*len(x); seen+=len(x)
        if batch_idx % PROGRESS_EVERY == 0 or batch_idx == len(train_loader):
            print(f'  epoch {epoch:02d} batch {batch_idx}/{len(train_loader)} | avg_loss={loss_sum/seen:.4f}')
    scheduler.step()
    yv,pv,_=predict(model,val_loader); mm=metric(yv,pv,0.5)
    hist.append({'epoch':epoch,'loss':loss_sum/seen,**mm})
    print(f"Epoch {epoch:02d} | loss={loss_sum/seen:.4f} | P={mm['P']:.5f} | A={mm['Accuracy']:.5f} | Se={mm['Sensitivity']:.5f} | Sp={mm['Specificity']:.5f} | F1={mm['F1']:.5f} | {time.time()-t0:.0f}s")
    if mm['P']>best+1e-6:
        best=mm['P']; best_epoch=epoch; stale=0; best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
    else:
        stale+=1
        if stale>=PATIENCE: print('Early stopping'); break
model.load_state_dict(best_state)
yv,pv,names=predict(model,val_loader)
fixed=metric(yv,pv,0.5)
scan=pd.DataFrame([metric(yv,pv,t) for t in np.round(np.arange(.05,.951,.005),3)])
row=scan.sort_values(['P','threshold'],ascending=[False,True]).iloc[0]
tuned=metric(yv,pv,float(row.threshold))
xpred=pd.DataFrame({'filename':names,'y_true':yv,'xception41p':pv})
xpred.to_csv(OUT/'xception41p_predictions.csv',index=False)
pd.DataFrame(hist).to_csv(OUT/'xception41p_history.csv',index=False)
scan.to_csv(OUT/'xception41p_threshold_scan.csv',index=False)
torch.save({'model':'xception41p.ra3_in1k','best_epoch':best_epoch,'state_dict':best_state},OUT/'xception41p_best.pt')
print('Xception @0.5:',fixed)
print('Xception tuned:',tuned)


## Diversity + simple fold-0 ensemble screen

In [ ]:
paths={
 'densenet':ROOT/'baseline_outputs'/'fold0_validation_predictions.csv',
 'convnext':ROOT/'screen_outputs'/'fold0_backbones'/'convnext_tiny_predictions.csv',
 'efficientnet':ROOT/'screen_outputs'/'fold0_backbones'/'efficientnet_b2_predictions.csv'}
for k,p in paths.items(): assert p.exists(),f'Missing {k}: {p}'

d=pd.read_csv(paths['densenet']).rename(columns={'p_pneumonia':'densenet'})[['filename','y_true','densenet']]
c=pd.read_csv(paths['convnext']).rename(columns={'p_pneumonia':'convnext'})[['filename','convnext']]
e=pd.read_csv(paths['efficientnet']).rename(columns={'p_pneumonia':'efficientnet'})[['filename','efficientnet']]
z=d.merge(c,on='filename').merge(e,on='filename').merge(xpred[['filename','xception41p']],on='filename')
assert len(z)==873

models_cols=['densenet','convnext','efficientnet','xception41p']
corr=z[models_cols].corr()
display(corr)
corr.to_csv(OUT/'probability_correlation.csv')

err={}
for col in models_cols:
    pred=(z[col].values>=0.5).astype(int); err[col]=set(np.where(pred!=z.y_true.values)[0])
overlap=[]
for i,a in enumerate(models_cols):
    for b in models_cols[i+1:]:
        inter=len(err[a]&err[b]); union=len(err[a]|err[b])
        overlap.append({'model_a':a,'model_b':b,'errors_a':len(err[a]),'errors_b':len(err[b]),'shared_errors':inter,'jaccard':inter/union if union else 0})
overlap_df=pd.DataFrame(overlap).sort_values('jaccard')
display(overlap_df)
overlap_df.to_csv(OUT/'error_overlap.csv',index=False)

# Coarse 2-model ensemble search: DenseNet + each candidate.
ens=[]
for other in ['convnext','efficientnet','xception41p']:
    for w in np.round(np.arange(0,1.001,.05),2):
        prob=w*z.densenet.values+(1-w)*z[other].values
        for t in np.round(np.arange(.20,.801,.01),2):
            mm=metric(z.y_true.values,prob,t)
            ens.append({'pair':f'densenet+{other}','dense_weight':float(w),'other_weight':float(1-w),**mm})
ens_df=pd.DataFrame(ens)
best_ens=ens_df.sort_values('P',ascending=False).groupby('pair',as_index=False).first().sort_values('P',ascending=False)
display(best_ens[['pair','dense_weight','other_weight','threshold','P','Accuracy','Sensitivity','Specificity','F1']])
ens_df.to_csv(OUT/'ensemble_grid.csv',index=False)
best_ens.to_csv(OUT/'best_pair_ensembles.csv',index=False)


In [ ]:
summary={'xception_model':'xception41p.ra3_in1k','image_size':299,'best_epoch':best_epoch,'metrics_at_0_5':fixed,'best_threshold':float(row.threshold),'tuned_metrics':tuned}
with open(OUT/'xception_summary.json','w',encoding='utf-8') as f: json.dump(summary,f,indent=2)
print(json.dumps(summary,indent=2))
print('\nBest pair ensembles:')
print(best_ens[['pair','dense_weight','other_weight','threshold','P']].to_string(index=False))
